In [1]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely.geometry import Point

# UMAP + PCA
import umap
from sklearn.decomposition import PCA

# Ensure we can import repo modules (main.py, etc.)
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
from main import Location2TextLightningModule

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
geoclip_ckpt_dir = "/home/libe2152/outputs/explainable-earth-embeddings/geoclip/pretrained/checkpoints"
geoclip_ckpt_path = Path(geoclip_ckpt_dir) / "location2text_pretrained.ckpt"

geoclip_model = Location2TextLightningModule.load_from_checkpoint(
    str(geoclip_ckpt_path),
    map_location=device,
    location_model_type="geoclip",
    location_model=None,
    location_model_filename=None,
    text_model_type="geoclip",
    text_model="geoclip",
    text_vocabulary="openai",
    finetune_mode="none",  # freeze all text model weights
    train_text_model=False
)
geoclip_model.to(device)

if geoclip_ckpt_path.exists():
    ckpt = torch.load(str(geoclip_ckpt_path), map_location=device, weights_only=False)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    geoclip_model.load_state_dict(state_dict, strict=False)
else:
    print(f"Checkpoint not found at {geoclip_ckpt_path}. Using GeoCLIP init weights only.")

geoclip_model = geoclip_model.to(device).eval()
print("device:", device)
print("output_dim:", geoclip_model.output_dim)

train_text_model False


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[LocationEmbeddingModel] GeoCLIP (or non-SatCLIP) backend: using locations as [lat, lon] without reordering. Example[0]=[0.0, 0.0]


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cuda
output_dim: 512


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
satclip_ckpt_dir = "/home/libe2152/outputs/explainable-earth-embeddings/satclip/geoyfcc-text/checkpoints"
satclip_ckpt_path = Path(satclip_ckpt_dir) / "Text2Location-epoch=46-val_loss=1.3700.ckpt"  # adjust if needed

satclip_model = Location2TextLightningModule.load_from_checkpoint(
    str(satclip_ckpt_path),
    map_location=device,
)
satclip_model.to(device)

if satclip_ckpt_path.exists():
    ckpt = torch.load(str(satclip_ckpt_path), map_location=device, weights_only=False)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    satclip_model.load_state_dict(state_dict, strict=False)
else:
    print(f"Checkpoint not found at {satclip_ckpt_path}. Using GeoCLIP init weights only.")

satclip_model = satclip_model.to(device).eval()
print("device:", device)
print("output_dim:", satclip_model.output_dim)

train_text_model True
using pretrained moco vit16
[LocationEmbeddingModel] SatCLIP backend: interpreting inputs as [lat, lon] and reordering to [lon, lat]. Example before[0]=[0.0, 0.0], after[0]=[0.0, 0.0]


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cuda
output_dim: 256


# Create concept embeddings with GeoCLIP and SatCLIP trained text encoders

In [7]:
import json
from pathlib import Path

In [6]:
@torch.no_grad()
def embed_texts(model, texts, device, batch_size=256, normalize=True):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding texts"):
        chunk = texts[i:i+batch_size]
        emb = model.text_model_predict(chunk, normalize=normalize)
        if emb.ndim == 1:
            emb = emb.unsqueeze(0)
        out.append(emb)
    return torch.cat(out, dim=0)

In [8]:
# Load geospatial concepts (list of strings)
vocab_dir = "/home/libe2152/projects/explainable-earth-embeddings/0_vocabs"  # adjust
geo_path = Path(vocab_dir) / "geoyfcc_concept_set.json"
with geo_path.open("r", encoding="utf-8") as f:
    geo_data = json.load(f)

if isinstance(geo_data, dict):
    geo_texts = list(geo_data.keys())
else:
    geo_texts = [str(x) for x in geo_data]

print(f"# geospatial concepts: {len(geo_texts)}")

# geospatial concepts: 10000


In [ ]:
# Embed them with the same model + preprocessing you used for desc_emb
geoclip_concept_emb = embed_texts(geoclip_model, geo_texts, device=device, batch_size=256, normalize=True).cpu()

print("geoclip_concept_emb:", geoclip_concept_emb.shape)

Embedding texts: 100%|██████████| 40/40 [00:01<00:00, 29.70it/s]

geo_emb: torch.Size([10000, 512])


In [10]:
# Embed them with the same model + preprocessing you used for desc_emb
satclip_concept_emb = embed_texts(satclip_model, geo_texts, device=device, batch_size=256, normalize=True).cpu()

print("satclip_concept_emb:", satclip_concept_emb.shape)

Embedding texts: 100%|██████████| 40/40 [00:01<00:00, 32.02it/s]

satclip_concept_emb: torch.Size([10000, 256])


# Hypothesis: GeoCLIP concept embeddings are highly aligned with their mean and lie in a low rank subspace, while SatCLIP embeddings are not stronly aligned with their mean and live in a higher rank subspace

## Checking alignment with their mean

In [11]:
geoclip_concept_emb_normalized = F.normalize(geoclip_concept_emb, dim=1)
satclip_concept_emb_normalized = F.normalize(satclip_concept_emb, dim=1)

In [12]:
geoclip_concept_mu = geoclip_concept_emb_normalized.mean(dim=0, keepdim=True)
geoclip_concept_mu = F.normalize(geoclip_concept_mu, dim=1)

cos_sim = (geoclip_concept_emb_normalized @ geoclip_concept_mu.T).squeeze()

print("Mean cosine similarity for geoclip concepts:", cos_sim.mean().item())
print("Std cosine similarity for geoclip concepts:", cos_sim.std().item())
print("Min cosine similarity for geoclip concepts:", cos_sim.min().item())

Mean cosine similarity for geoclip concepts: 0.8399471640586853
Std cosine similarity for geoclip concepts: 0.08517836034297943
Min cosine similarity for geoclip concepts: 0.23641015589237213


This supports our hypothesis about GeoCLIP -- the concept vectors are strongly aligned with the mean vector, and the low STD shows all vectors behave similarly (there is this strong global direction)

In [13]:
satclip_concept_mu = satclip_concept_emb_normalized.mean(dim=0, keepdim=True)
satclip_concept_mu = F.normalize(satclip_concept_mu, dim=1)

cos_sim = (satclip_concept_emb_normalized @ satclip_concept_mu.T).squeeze()

print("Mean cosine similarity for satclip concepts:", cos_sim.mean().item())
print("Std cosine similarity for satclip concepts:", cos_sim.std().item())
print("Min cosine similarity for satclip concepts:", cos_sim.min().item())

Mean cosine similarity for satclip concepts: 0.841179370880127
Std cosine similarity for satclip concepts: 0.1033058911561966
Min cosine similarity for satclip concepts: 0.17960338294506073


We see similar results for SatCLIP, so maybe SatCLIP is not lacking this strong global direction

# Hypothesis: GeoCLIP concept vectors live in a lower dimensional subspace, while SatCLIP concept vectors live in a higher dimensional subspace?

In [16]:
U, S, V = torch.pca_lowrank(geoclip_concept_emb_normalized, q=min(50, geoclip_concept_emb_normalized.shape[1]))
var_explained = S / S.sum()

print("Top 10 variance ratios for geoclip concepts:", var_explained[:10])
print("Cumulative variance (top 1) for geoclip concepts:", var_explained[0].item())
print("Cumulative variance (top 5) for geoclip concepts:", var_explained[:5].sum().item())

Top 10 variance ratios for geoclip concepts: tensor([0.0591, 0.0426, 0.0351, 0.0344, 0.0291, 0.0278, 0.0262, 0.0249, 0.0243,
        0.0239])
Cumulative variance (top 1) for geoclip concepts: 0.05913073942065239
Cumulative variance (top 5) for geoclip concepts: 0.20034825801849365


In [17]:
entropy = -(var_explained * torch.log(var_explained + 1e-8)).sum()
effective_rank = torch.exp(entropy)

print("Effective rank for geoclip concepts:", effective_rank.item())

Effective rank for geoclip concepts: 46.69618606567383


In [18]:
U, S, V = torch.pca_lowrank(satclip_concept_emb_normalized, q=min(50, satclip_concept_emb_normalized.shape[1]))
var_explained = S / S.sum()

print("Top 10 variance ratios for satclip concepts:", var_explained[:10])
print("Cumulative variance (top 1) for satclip concepts:", var_explained[0].item())
print("Cumulative variance (top 5) for satclip concepts:", var_explained[:5].sum().item())

Top 10 variance ratios for satclip concepts: tensor([0.0464, 0.0417, 0.0340, 0.0324, 0.0305, 0.0300, 0.0297, 0.0273, 0.0266,
        0.0263])
Cumulative variance (top 1) for satclip concepts: 0.046449095010757446
Cumulative variance (top 5) for satclip concepts: 0.18498146533966064


In [19]:
entropy = -(var_explained * torch.log(var_explained + 1e-8)).sum()
effective_rank = torch.exp(entropy)

print("Effective rank for satclip concepts:", effective_rank.item())

Effective rank for satclip concepts: 46.61178207397461


# Hypothesis: centering around the mean might destroy the signal (hence the UMAP observations)

In [20]:
geoclip_concept_emb_normalized_centered = geoclip_concept_emb_normalized - geoclip_concept_emb_normalized.mean(dim=0, keepdim=True)

orig_norm = geoclip_concept_emb_normalized.norm(dim=1).mean()
centered_norm = geoclip_concept_emb_normalized_centered.norm(dim=1).mean()

print("GeoCLIP concepts Norm before:", orig_norm.item())
print("GeoCLIP concepts Norm after centering:", centered_norm.item())
print("GeoCLIP concepts Ratio:", (centered_norm / orig_norm).item())

GeoCLIP concepts Norm before: 1.0
GeoCLIP concepts Norm after centering: 0.5288249254226685
GeoCLIP concepts Ratio: 0.5288249254226685


In [21]:
satclip_concept_emb_normalized_centered = satclip_concept_emb_normalized - satclip_concept_emb_normalized.mean(dim=0, keepdim=True)

orig_norm = satclip_concept_emb_normalized.norm(dim=1).mean()
centered_norm = satclip_concept_emb_normalized_centered.norm(dim=1).mean()

print("SatCLIP concepts Norm before:", orig_norm.item())
print("SatCLIP concepts Norm after centering:", centered_norm.item())
print("SatCLIP concepts Ratio:", (centered_norm / orig_norm).item())

SatCLIP concepts Norm before: 1.0
SatCLIP concepts Norm after centering: 0.5236718654632568
SatCLIP concepts Ratio: 0.5236718654632568


In [22]:
sim = geoclip_concept_emb_normalized @ geoclip_concept_emb_normalized.T
print("GeoCLIP Mean pairwise similarity:", sim.mean().item())
print("GeoCLIP Std pairwise similarity:", sim.std().item())

GeoCLIP Mean pairwise similarity: 0.7055113315582275
GeoCLIP Std pairwise similarity: 0.10788492858409882


In [23]:
sim = satclip_concept_emb_normalized @ satclip_concept_emb_normalized.T
print("SatCLIP Mean pairwise similarity:", sim.mean().item())
print("SatCLIP Std pairwise similarity:", sim.std().item())

SatCLIP Mean pairwise similarity: 0.7075827121734619
SatCLIP Std pairwise similarity: 0.13129444420337677


# Hypothesis: check residuals with respect to mean

In [27]:
geoclip_concept_emb_normalized = F.normalize(geoclip_concept_emb_normalized, dim=1)
mu = geoclip_concept_emb_normalized.mean(dim=0, keepdim=True)

residuals = geoclip_concept_emb_normalized - mu
print("GeoCLIP residuals norm: ", residuals.norm(dim=1).mean().item())
print("GeoCLIP Residual/original ratio:", (residuals.norm(dim=1).mean() / geoclip_concept_emb_normalized.norm(dim=1).mean()).item())

GeoCLIP residuals norm:  0.5288249254226685
GeoCLIP Residual/original ratio: 0.5288249254226685


In [28]:
satclip_concept_emb_normalized = F.normalize(satclip_concept_emb_normalized, dim=1)
mu = satclip_concept_emb_normalized.mean(dim=0, keepdim=True)

residuals = satclip_concept_emb_normalized - mu
print("SatCLIP residuals norm: ", residuals.norm(dim=1).mean().item())
print("SatCLIP Residual/original ratio:", (residuals.norm(dim=1).mean() / satclip_concept_emb_normalized.norm(dim=1).mean()).item())

SatCLIP residuals norm:  0.5236718654632568
SatCLIP Residual/original ratio: 0.5236718654632568
